<a href="https://colab.research.google.com/github/grullaandrea-png/Text-Mining/blob/main/RoBERT_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Named Entity Recognition (NER) with RoBERTa
In this section, we tackle the Named Entity Recognition (NER) task by implementing a model based on the **RoBERTa** (`roberta-base`) architecture. This step is necessary to overcome the limitations encountered with BERT, specifically the excessive fragmentation of Out-Of-Vocabulary (OOV) terms and its high sensitivity to text casing variations. Thanks to its Byte-Pair Encoding (BPE) tokenization, RoBERTa offers greater robustness on Out-Of-Distribution (OOD) texts.

We begin by installing the necessary libraries (including `seqeval` for sequence metrics) and importing the fundamental modules.

In [1]:
!pip install -q transformers datasets evaluate seqeval accelerate

from google.colab import drive
import os
import json
import random
import numpy as np
from datasets import Dataset, DatasetDict

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.2 MB/s eta 0:00:00


## Data Loading and Preparation
We proceed by loading the previously cleaned and re-tokenized dataset. The data is split into three partitions to ensure proper model evaluation:
- **Train set**: for training (approx. 70%).
- **Validation set**: for monitoring during training (15%).
- **Test set**: for final evaluation (15%).

Next, we map the BIO labels into dictionaries (`label2id` and `id2label`) and convert the data into the native `DatasetDict` format of the Hugging Face library, which is optimized for Transformer training.

In [3]:
drive.mount('/content/drive')
DATA_PATH = '/content/drive/MyDrive/DV-TM/DATA/'

def load_jsonl(path):
    with open(path, 'r') as f:
        return [json.loads(line) for line in f]

train_cleaned = load_jsonl(os.path.join(DATA_PATH, 'train_cleaned.jsonl'))
test_retokenized = load_jsonl(os.path.join(DATA_PATH, 'test_retokenized.jsonl'))

random.seed(42)
indices = list(range(len(train_cleaned)))
random.shuffle(indices)

n = len(train_cleaned)
n_val  = int(n * 0.15)
n_test = n_val
n_train = n - n_val - n_test

train_data = [train_cleaned[i] for i in indices[:n_train]]
val_data   = [train_cleaned[i] for i in indices[n_train:n_train + n_val]]
test_data  = [train_cleaned[i] for i in indices[n_train + n_val:n_train + n_val + n_test]]

unique_labels = set()
for item in train_cleaned:
    for label in item['labels']:
        unique_labels.add(label)

label_list = sorted(list(unique_labels))
if 'O' in label_list:
    label_list.remove('O')
    label_list.insert(0, 'O')

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

# Creazione Dataset Hugging Face
def create_hf_dataset(data):
    return Dataset.from_dict({
        "id": [item["id"] for item in data],
        "tokens": [item["tokens"] for item in data],
        "ner_tags": [[label2id[label] for label in item["labels"]] for item in data]
    })

datasets = DatasetDict({
    "train": create_hf_dataset(train_data),
    "validation": create_hf_dataset(val_data),
    "test": create_hf_dataset(test_data)
})

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Tokenization and Label Alignment
This step is crucial. Since we are dealing with a Token Classification task and providing the model with lists of pre-tokenized words, it is essential to initialize the RoBERTa tokenizer with the `add_prefix_space=True` parameter. This ensures that the BPE tokenization correctly recognizes word boundaries.

Furthermore, we define the `tokenize_and_align_labels` function to resolve the misalignment caused by sub-tokens. We assign the actual NER label to the first sub-token of each word and the index `-100` to subsequent sub-tokens and special tokens. The `-100` index is automatically ignored by PyTorch's Cross-Entropy Loss function during error calculation.

In [4]:
from transformers import AutoTokenizer

model_checkpoint = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, add_prefix_space=True)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = datasets.map(tokenize_and_align_labels, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/2200 [00:00<?, ? examples/s]

Map:   0%|          | 0/471 [00:00<?, ? examples/s]

Map:   0%|          | 0/471 [00:00<?, ? examples/s]

## Model Setup, Metrics, and Fine-Tuning
We initialize the `AutoModelForTokenClassification` architecture based on the pre-trained weights of `roberta-base`, correctly configuring the number of labels for our specific domain.

We then define the function to calculate the evaluation metrics using the `seqeval` library (Precision, Recall, F1-Score, and Accuracy), which computes performance by safely excluding padding tokens (`-100`). Finally, we configure the training hyperparameters using the `Trainer` API (learning rate of 2e-5, batch size of 16, and 3 training epochs).

In [5]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification
import evaluate

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

training_args = TrainingArguments(
    output_dir="./roberta-ner-model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=10
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Train start...")
trainer.train()

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.weight               | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Train start...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.002184,0.000688,1.000000,1.000000,1.000000,1.000000
2,0.001108,0.000357,1.000000,1.000000,1.000000,1.000000
3,0.000939,0.000298,1.000000,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=414, training_loss=0.07809637862163177, metrics={'train_runtime': 52.3207, 'train_samples_per_second': 126.145, 'train_steps_per_second': 7.913, 'total_flos': 71514098943120.0, 'train_loss': 0.07809637862163177, 'epoch': 3.0})

## Evaluation and Out-Of-Distribution (OOD) Stress Testing
After training, we evaluate the model's generalization capabilities in two phases:
1. **Test Set Metrics**: We extract the complete classification report on data not seen during training.
2. **Qualitative Stress Test**: We use a NER `pipeline` to subject the model to tricky sentences, written in lowercase or containing rare syntactic and lexical patterns (OOV). This test aims to demonstrate how RoBERTa's BPE tokenization significantly mitigates the "brittleness" previously observed, showing greater flexibility to text casing variations.

In [7]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from transformers import pipeline

print("Test Set precictions...")
predictions_output = trainer.predict(tokenized_datasets["test"])
predictions = np.argmax(predictions_output.predictions, axis=2)
labels = predictions_output.label_ids

true_labels_flat = []
pred_labels_flat = []

for pred_seq, label_seq in zip(predictions, labels):
    for pred, label in zip(pred_seq, label_seq):
        if label != -100:
            true_labels_flat.append(label_list[label])
            pred_labels_flat.append(label_list[pred])

print("\n--- Classificazion metrics RoBERTa ---")
report = classification_report(
    true_labels_flat,
    pred_labels_flat,
    target_names=label_list,
    labels=label_list
)
print(report)

# Stress Test OOD
ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

stress_tests = [
    "Join CyberNinja Corp today ! Our team needs a Senior Cloud Security Engineer mastering AWS .",
    "london data scientist wanted at deepmind . heavy matlab background is strictly required .",
    "Mastered Swift ? Relocate to Cupertino ! Apple is hunting for a skilled iOS Developer ."
]

print("\n--- TRESS TEST ROBERTA RESULTS---")
for i, text in enumerate(stress_tests, 1):
    print(f"\nTest {i}: {text}")
    entities = ner_pipeline(text)
    if not entities:
        print("  [No entity found]")
    else:
        for ent in entities:
            print(f"  - Entity: '{ent['word']}' | Label: {ent['entity_group']} | Score: {ent['score']:.4f}")

Test Set precictions...



--- Classificazion metrics RoBERTa ---
              precision    recall  f1-score   support

           O       1.00      1.00      1.00      4051
   B-COMPANY       1.00      1.00      1.00       471
  B-JOBTITLE       1.00      1.00      1.00       471
  B-LOCATION       1.00      1.00      1.00       471
     B-SKILL       1.00      1.00      1.00       471
   I-COMPANY       1.00      1.00      1.00       178
  I-JOBTITLE       1.00      1.00      1.00       471
  I-LOCATION       1.00      1.00      1.00       129
     I-SKILL       1.00      1.00      1.00       179

    accuracy                           1.00      6892
   macro avg       1.00      1.00      1.00      6892
weighted avg       1.00      1.00      1.00      6892


--- TRESS TEST ROBERTA RESULTS---

Test 1: Join CyberNinja Corp today ! Our team needs a Senior Cloud Security Engineer mastering AWS .
  - Entity: ' Cyber' | Label: COMPANY | Score: 0.9989
  - Entity: 'Ninja Corp' | Label: COMPANY | Score: 0.9724
  - En

### Conclusions and Critical Analysis of Results
The output of the stress tests confirms that the previously discussed architectural limitations have been successfully overcome:
- **Casing Handling**: In *Test 2*, RoBERTa successfully recognizes geographic ("london") and corporate ("deepmind") entities written entirely in lowercase, unlike the previous BERT model which relied heavily on uppercase letters.
- **Sub-token Robustness**: In *Test 1*, the model recognizes the entire compound entity "CyberNinja Corp". This demonstrates that the pipeline now successfully aggregates BPE sub-tokens even on out-of-vocabulary and entirely fictional terms.
- **Semantic Ambiguity**: The only marginal false positive/negative (e.g., "Apple" predicted with low confidence as a skill in *Test 3*) highlights a natural semantic ambiguity on polysemous terms. However, the overall performance indicates a clear leap in linguistic abstraction capabilities and real-world robustness.